# MATH1604 — Analysis of Python Quiz Responses
## Review Notebook: Team Member 4 (Team Leader)

**Author:** Danish Imran Agus Bin Faisal  
**Student ID:** 202023762  
**Role:** Team Member 4 — Integration & Execution / Team Leader  
**Module:** MATH1604 Modelling for Big Data  
**Date:** May 2025

---

## 1. Introduction

This notebook documents my individual contribution to the MATH1604 group project, which involved analysing a dataset of 64 respondents' answers to a 100-question multiple-choice Python quiz. Each question has four answer options (1–4), and the central hypothesis is that the quiz setter arranged the correct answers in a deliberate, non-random pattern.

As Team Member 4 and Team Leader, my responsibilities were:

1. **Creating and managing the GitHub repository**, including branch strategy, pull request reviews, and commit history.
2. **Writing `data_extraction_M1.py`** — the parsing module that reads raw quiz answer files and extracts structured answer sequences.
3. **Writing `run_full_analysis_M4.py`** — the integration script that orchestrates the full pipeline from download through to visualisation.

This notebook walks through my modules, demonstrates their functionality, and presents my investigation into the pattern detection task.

---
## 2. Repository Structure & GitHub Management

As Team Leader I set up the following repository structure, which all team members contributed to via feature branches and pull requests:

```
MATH1604-group-project/
├── data/                          # Downloaded respondent answer files
│   ├── answers_respondent_1.txt
│   └── ... (up to answers_respondent_64.txt)
├── scripts/
│   ├── data_extraction_M1.py      # Team Member 1 (me)
│   ├── data_preparation_M2.py     # Team Member 2
│   ├── data_analysis_M3.py        # Team Member 3
│   └── run_full_analysis_M4.py    # Team Member 4 (me)
├── output/
│   └── collated_answers.txt
└── reviews/
    ├── review_by_M1.ipynb
    ├── review_by_M2.ipynb
    ├── review_by_M3.ipynb
    └── review_by_M4.ipynb         # This notebook
```

**Branch strategy used:**
- `main` — protected; only merged via pull requests
- `feature/m1-extraction` — my extraction module
- `feature/m4-integration` — my integration script
- `feature/m2-download` — teammate's download module
- `feature/m3-analysis` — teammate's analysis module

All merges were done through pull requests with code review comments.

---
## 3. Module M1 — Data Extraction Walkthrough

### 3.1 Purpose

`data_extraction_M1.py` provides two functions:

- **`extract_answers_sequence(file_path)`** — reads a raw quiz answer `.txt` file and returns a list of 100 integers (1–4 for selected answers, 0 for unanswered).
- **`write_answers_sequence(answers, n, destination_path)`** — writes that list to a text file for downstream use.

### 3.2 How the parser works

Each quiz file has a repeating structure for each question:

```
Question N. [question text]
[ ] Answer N.1
[x] Answer N.2     ← selected
[ ] Answer N.3
[ ] Answer N.4
```

The parser iterates line-by-line. When it detects a `Question N.` header, it processes the accumulated option lines from the *previous* question (if any), determines which option has `[x]`, and appends that position (1–4) to the sequence. If no `[x]` is found, it appends 0.

In [ ]:
import sys
import os

# Add the scripts directory to the path so we can import our modules
sys.path.insert(0, os.path.join('..', 'scripts'))

from data_extraction_M1 import extract_answers_sequence, write_answers_sequence

print('data_extraction_M1 imported successfully.')

In [ ]:
# Demonstrate extract_answers_sequence on the first respondent file
data_folder = os.path.join('..', 'data')
file_path = os.path.join(data_folder, 'answers_respondent_1.txt')

answers_r1 = extract_answers_sequence(file_path)

print(f'Type   : {type(answers_r1)}')
print(f'Length : {len(answers_r1)}')
print(f'First 20 answers: {answers_r1[:20]}')
print(f'Unique values found: {sorted(set(answers_r1))}')

### 3.3 Checking for unanswered questions

In [ ]:
unanswered = [i+1 for i, a in enumerate(answers_r1) if a == 0]

if unanswered:
    print(f'Respondent 1 left {len(unanswered)} question(s) unanswered: questions {unanswered}')
else:
    print('Respondent 1 answered all 100 questions.')

### 3.4 Demonstrating write_answers_sequence

In [ ]:
output_folder = os.path.join('..', 'output')
os.makedirs(output_folder, exist_ok=True)

write_answers_sequence(answers_r1, 1, output_folder)

# Verify the written file
written_file = os.path.join(output_folder, 'answers_list_respondent_1.txt')
with open(written_file, 'r') as f:
    lines = f.readlines()

print(f'Written file has {len(lines)} lines (expected 100).')
print(f'First 10 lines: {[l.strip() for l in lines[:10]]}')

### 3.5 Error handling demonstration

In [ ]:
# Test that FileNotFoundError is raised correctly
try:
    extract_answers_sequence('nonexistent_file.txt')
except FileNotFoundError as e:
    print(f'Correctly raised FileNotFoundError: {e}')

# Test that ValueError is raised for wrong-length lists
try:
    write_answers_sequence([1, 2, 3], 1, output_folder)
except ValueError as e:
    print(f'Correctly raised ValueError: {e}')

---
## 4. Module M4 — Integration Pipeline Walkthrough

### 4.1 Purpose

`run_full_analysis_M4.py` ties all three other modules together into a single executable pipeline:

| Step | Function called | Module |
|------|----------------|--------|
| 1 | Ensure folders exist | M4 |
| 2 | `download_answer_files(...)` | M2 |
| 3 | `collate_answer_files(...)` | M2 |
| 4 | `extract_answers_sequence(...)` + `write_answers_sequence(...)` | M1 |
| 5 | `generate_means_sequence(...)` | M3 |
| 6 | `visualize_data(..., 1)` and `visualize_data(..., 2)` | M3 |

### 4.2 Running the full pipeline

In [ ]:
from data_preparation_M2 import download_answer_files, collate_answer_files
from data_analysis_M3 import generate_means_sequence, visualize_data

BASE_URL = 'https://raw.githubusercontent.com/fc-leeds/MATH1604_2025_2026_data/main'
DATA_FOLDER = os.path.join('..', 'data')
OUTPUT_FOLDER = os.path.join('..', 'output')
COLLATED_FILE = os.path.join(OUTPUT_FOLDER, 'collated_answers.txt')

print('All modules imported successfully. Ready to run pipeline.')

In [ ]:
# Step 1 — Download (requesting 70 to test robustness; only 64 exist)
download_answer_files(BASE_URL, DATA_FOLDER, 70)

In [ ]:
# Step 2 — Collate
collate_answer_files(DATA_FOLDER)

# Verify output
size = os.path.getsize(COLLATED_FILE)
print(f'collated_answers.txt created — size: {size:,} bytes')

In [ ]:
# Step 3 — Extract all sequences and count files processed
data_files = sorted(
    [f for f in os.listdir(DATA_FOLDER) if f.startswith('answers_respondent_') and f.endswith('.txt')],
    key=lambda x: int(x.replace('answers_respondent_', '').replace('.txt', ''))
)

all_sequences = []
for fname in data_files:
    fp = os.path.join(DATA_FOLDER, fname)
    n = int(fname.replace('answers_respondent_', '').replace('.txt', ''))
    seq = extract_answers_sequence(fp)
    all_sequences.append(seq)

print(f'Successfully extracted {len(all_sequences)} respondent sequences.')

---
## 5. Pattern Investigation

### 5.1 Computing means per question

If the correct answers were assigned randomly, we would expect the mean answer value across respondents per question to be approximately 2.5 (the midpoint of options 1–4). A deliberate pattern would likely cause the means to deviate from this or show a repeating structure.

In [ ]:
means = generate_means_sequence(COLLATED_FILE)

print(f'Number of means computed: {len(means)}')
print(f'First 20 means: {[round(m, 3) for m in means[:20]]}')
print(f'Overall mean of means: {round(sum(means)/len(means), 4)}')
print(f'Min mean: {round(min(means), 4)} | Max mean: {round(max(means), 4)}')

### 5.2 Scatter plot — means per question

In [ ]:
visualize_data(COLLATED_FILE, 1)

### 5.3 Line plot — all individual respondent sequences

In [ ]:
visualize_data(COLLATED_FILE, 2)

### 5.4 Frequency analysis — how often each answer (1–4) appears per question

In [ ]:
import matplotlib.pyplot as plt

# Count frequency of each answer option (1–4) per question
freq = {1: [], 2: [], 3: [], 4: []}

for q_idx in range(100):
    col = [seq[q_idx] for seq in all_sequences if seq[q_idx] != 0]
    total = len(col)
    for opt in [1, 2, 3, 4]:
        freq[opt].append(col.count(opt) / total if total > 0 else 0)

questions = list(range(1, 101))
colours = {1: '#e74c3c', 2: '#3498db', 3: '#2ecc71', 4: '#f39c12'}

fig, ax = plt.subplots(figsize=(15, 5))
for opt in [1, 2, 3, 4]:
    ax.plot(questions, freq[opt], label=f'Answer {opt}',
            color=colours[opt], linewidth=0.8, alpha=0.8)

ax.axhline(0.25, color='black', linestyle='--', linewidth=1,
           label='Expected if random (0.25)')
ax.set_title('Proportion of Respondents Selecting Each Answer Option per Question')
ax.set_xlabel('Question Number')
ax.set_ylabel('Proportion of respondents')
ax.set_xlim(0.5, 100.5)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.5 Investigating periodicity

A common quiz-setting strategy is to cycle through answers in a repeating pattern (e.g. 1, 2, 3, 4, 1, 2, 3, 4, ...). We can investigate this by checking what the most frequently selected answer is for each question and seeing whether it repeats with a regular period.

In [ ]:
# Find the modal (most common) answer per question
from collections import Counter

modal_answers = []
for q_idx in range(100):
    col = [seq[q_idx] for seq in all_sequences if seq[q_idx] != 0]
    if col:
        modal = Counter(col).most_common(1)[0][0]
    else:
        modal = 0
    modal_answers.append(modal)

print('Modal (most common) answer per question:')
print(modal_answers)
print()
print('First 20:', modal_answers[:20])
print('Questions 21-40:', modal_answers[20:40])

In [ ]:
# Plot modal answers to look for a visual repeating pattern
fig, ax = plt.subplots(figsize=(15, 4))
ax.scatter(questions, modal_answers, color='darkblue', s=40, zorder=3)
ax.plot(questions, modal_answers, color='steelblue', linewidth=0.8, alpha=0.6)

ax.set_title('Most Frequently Selected Answer per Question (Modal Answer)')
ax.set_xlabel('Question Number')
ax.set_ylabel('Modal Answer (1–4)')
ax.set_xlim(0.5, 100.5)
ax.set_yticks([1, 2, 3, 4])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Test for period-4 pattern: does the modal answer repeat every 4 questions?
period = 4
matches = sum(1 for i in range(100 - period) if modal_answers[i] == modal_answers[i + period])
total_comparisons = 100 - period

print(f'Period-{period} consistency check:')
print(f'  Matching pairs: {matches} / {total_comparisons}')
print(f'  Match rate: {matches/total_comparisons:.1%}')
print()

# Also test period 5 and period 10 as alternatives
for p in [2, 3, 5, 10, 20, 25]:
    m = sum(1 for i in range(100 - p) if modal_answers[i] == modal_answers[i + p])
    print(f'Period-{p:2d}: {m}/{100-p} matches ({m/(100-p):.1%})')

### 5.6 Statistical test — chi-squared goodness of fit

We can run a chi-squared test on the overall distribution of answers across all questions. If the correct answers were truly random and uniform, each option (1–4) should appear in roughly 25% of questions. A significant result suggests a non-random structure.

In [ ]:
from scipy import stats

# Count how often each modal answer appears across all 100 questions
modal_counts = Counter(modal_answers)
print('Modal answer distribution across 100 questions:')
for opt in [1, 2, 3, 4]:
    print(f'  Answer {opt}: {modal_counts.get(opt, 0)} questions')

observed = [modal_counts.get(opt, 0) for opt in [1, 2, 3, 4]]
expected = [25, 25, 25, 25]  # Equal distribution under null hypothesis

chi2_stat, p_value = stats.chisquare(f_obs=observed, f_exp=expected)

print(f'\nChi-squared statistic: {chi2_stat:.4f}')
print(f'p-value: {p_value:.4f}')

alpha = 0.05
if p_value < alpha:
    print(f'\nResult: p < {alpha} → Reject H₀. The modal answer distribution is NOT uniform.')
    print('This supports the hypothesis that a deliberate pattern exists.')
else:
    print(f'\nResult: p ≥ {alpha} → Fail to reject H₀. No strong evidence of non-uniformity.')

---
## 6. Summary of Findings

*(Fill in this section once you have run the notebook and seen the actual results. Use the template below as a guide.)*

Based on the analysis carried out in this notebook, the following conclusions can be drawn:

**Was a pattern detected?**  
[Describe what the scatter plot and modal answer plot revealed. E.g.: "The modal answer per question appears to cycle through values 1, 2, 3, 4 with a period of 4, suggesting the quiz setter used a simple rotating answer key." OR "No clear repeating pattern was detected in the means sequence, though the chi-squared test suggests the distribution is non-uniform."]

**What does the chi-squared test tell us?**  
[Report your p-value and what conclusion you draw.]

**What does the period test tell us?**  
[Report the period-4 (or whichever period) match rate and what it implies.]

---
## 7. Limitations and Assumptions

**Assumption 1 — The modal answer represents the correct answer.**  
This analysis assumes that the most commonly selected answer per question is likely to be the correct one. This is only valid if the majority of respondents are selecting the right answer, which may not hold if respondents are guessing randomly or if the quiz is very difficult.

**Assumption 2 — Respondents behave independently.**  
The statistical tests assume that respondents answered independently. If collaboration occurred, the frequency distributions would be artificially skewed.

**Assumption 3 — Unanswered questions are missing at random.**  
Questions coded as 0 (unanswered) are excluded from means calculations. If certain questions are systematically skipped (e.g. because they are harder or appear at the end), this exclusion may bias the means.

**Limitation 1 — Small sample size.**  
With only 64 respondents, per-question frequencies are noisy. A dominant correct answer may not clearly emerge for harder questions where responses are spread across all four options.

**Limitation 2 — Pattern identification is correlational.**  
Even if a period-4 pattern is found in the modal answers, this does not definitively confirm that the quiz setter used that pattern — it could arise by chance in a small dataset.

---
## 8. Conclusion

This notebook has demonstrated the full functionality of my two assigned modules (`data_extraction_M1.py` and `run_full_analysis_M4.py`), verified their correctness with test cases and edge case handling, and applied statistical and visual methods to investigate the pattern hypothesis.

[Write 2–3 sentences summarising your actual finding once you have run the code and seen the real results.]

The pipeline is modular, well-documented, and designed to function independently of other team members' code through the use of the provided `.pyc` reference files during development and integration testing.